In [20]:
%%capture --no-stderr
%pip install openai

In [21]:
import chromadb
from chromadb.utils import embedding_functions
import openai
from openai import OpenAI

In [22]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_key,
    model_name="text-embedding-3-small"
)

In [23]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="CVs",
    embedding_function=openai_ef
)

documents = [
    "Esperto in Digital Marketing e Social Media Strategy. Gestisce campagne pubblicitarie, promozione di prodotti e brand awareness.",
    "Sviluppatore Full Stack specializzato in Python, Django e React. Creazione di architetture web avanzate e database.",
    "Graphic Designer e Content Creator. Specializzato in brand identity, creazione di contenuti visivi e materiale promozionale."
]

metadatas = [
    {"source": "CV_Esperto_Marketing.txt"},
    {"source": "CV_Sviluppatore_Web.txt"},
    {"source": "CV_Graphic_Designer.txt"}
]

ids = ["id_1", "id_2", "id_3"]

In [24]:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

In [25]:
user_question = "mi serve qualcuno per promuovere il mio prodotto"

results = collection.query(
    query_texts=[user_question],
    n_results=2
)

In [26]:
context = f"CONTESTO: nome file {results['metadatas'][0][0]['source']} ecco il paragrafo più significativo: {results['documents'][0][0]}"

prompt = f"""Dato il seguente contesto {context} rispondi alla domanda dell'utente {user_question} 
spiegando che nel file individuato c'è il profilo più adatto.
Argomenta la scelta utilizzando il contenuto del testo individuato nel contesto"""

In [27]:
client = OpenAI(api_key=openai_key)

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "developer", 
            "content": "Sei un assistente HR, specializzato nella ricerca di profili professionali"
        },  
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(completion.choices[0].message.content)

Nel contesto fornito, il file "CV_Esperto_Marketing.txt" descrive un professionista altamente qualificato nel settore del Digital Marketing e della Social Media Strategy. Questa persona gestisce campagne pubblicitarie e si occupa della promozione di prodotti, nonché della brand awareness. 

Queste competenze sono esattamente ciò che serve per promuovere efficacemente il tuo prodotto. La capacità di gestire campagne pubblicitarie indica che il candidato sa come raggiungere il pubblico desiderato e massimizzare l'efficacia della comunicazione promozionale. Inoltre, la competenza nella promozione di prodotti sottolinea la sua esperienza nel mettere in risalto le caratteristiche uniche del tuo prodotto per attrarre clienti. Infine, la specializzazione nella brand awareness garantisce che il tuo prodotto non solo venga conosciuto, ma anche associato positivamente nel lungo termine con il tuo brand. Questi elementi rendono questo professionista il candidato ideale per promuovere il tuo prodo